# Assignment 1: ResNet Classifier

In this notebook, you will implement a ResNet-based image classifier.  

In [ ]:
! pip install -q git+https://github.com/ontariotechu-csci4052u/csci4052u
import csci4052u
print("CSCI 4052U version:", csci4052u.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.7/228.7 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 58.3 MB/s eta 0:00:00
CSCI 4052U version: 2026-03-19


## Dataset

In [ ]:
# @title Load the Lightning Data Module.
fraction = 0.1 # @param {"type":"number","placeholder":"Fraction of the data to be used."}
input_shape_str = "3x224x224" # @param ["1x28x28","3x224x224"]
input_shape = (3, 224, 224) if input_shape_str == "3x224x224" else None

data = csci4052u.data.FashionMNISTDataModule(
    fraction=fraction,
    shape=input_shape,
)
data.prepare_data()

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 202kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.82MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 21.1MB/s]


## Lightning Module

Important:

- You should use `models.resnet18(...)` as the backbone of your model.
- You must load the pretrained model weights from `models.ResNet18_Weights.DEFAULT`.  This will work much better than `resnet18(pretrained=True)`.
- You **must** use the weight specific transformation to preprocess each batch so that they are normalized using the pretrained mean and std.  See `models.ResNet18_Weights.DEFAULT.transforms()`.
- You **must** freeze the layers of `resnet18` backbone so that they are no longer adjusted during training.
- You **must** replace the backbone classifier head (`resnet_model.fc`) with a linear classifier that outputs logits in $\mathbb{R}^{10}$ to correspond to the FashionMNIST dataset.

In [ ]:
from lightning import LightningModule
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchmetrics

class ResNetClassifier(LightningModule):
  def __init__(self, lr: float = 1e-3):
    super().__init__()
    self.save_hyperparameters()
    self.lr = lr

    # 1. Load pretrained ResNet-18 weights
    weights = models.ResNet18_Weights.DEFAULT
    self.backbone = models.resnet18(weights=weights)

    # 2. Store the weight-specific transform for preprocessing
    self.transforms = weights.transforms()

    # 3. Freeze all backbone layers
    for param in self.backbone.parameters():
        param.requires_grad = False

    # 4. Replace the classifier head with a linear layer for 10 classes
    num_features = self.backbone.fc.in_features
    self.backbone.fc = nn.Linear(num_features, 10)

    # 5. Initialize Torchmetrics to cleanly track performance metrics
    self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10)
    self.val_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10)
    self.test_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10)

  def forward(self, x):
    # Fix: Expand 1-channel grayscale to 3-channel RGB without allocating new memory
    if x.ndim == 4 and x.shape[1] == 1:
        x = x.expand(-1, 3, -1, -1)
    elif x.ndim == 3 and x.shape[0] == 1:
        x = x.expand(3, -1, -1)

    # Preprocess/scale x for resnet18 (handles interpolation to 224x224 and normalization)
    x = self.transforms(x)
    return self.backbone(x)

  def training_step(self, batch, batch_idx):
    x, y = batch
    logits = self(x)
    loss = F.cross_entropy(logits, y)

    self.train_acc(logits, y)
    self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log("train_acc", self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def validation_step(self, batch, batch_idx):
    x, y = batch
    logits = self(x)
    loss = F.cross_entropy(logits, y)

    self.val_acc(logits, y)
    self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log("val_acc", self.val_acc, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def test_step(self, batch, batch_idx):
    x, y = batch
    logits = self(x)
    loss = F.cross_entropy(logits, y)

    self.test_acc(logits, y)
    self.log("test_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log("test_acc", self.test_acc, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def configure_optimizers(self):
    # Strict adherence: Optimize only the final classification head layer parameters
    return torch.optim.Adam(self.backbone.fc.parameters(), lr=self.lr)


## Training

In [ ]:
csci4052u.training.train(
    model_name="ResNet",
    model=ResNetClassifier(),
    datamodule=data,
    max_epochs=5,
    monitor_metric="val_loss",
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 264MB/s]
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: True, using: 1 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: True, using: 1 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone   │ ResNet              │ 11.2 M │ train │     0 │
│ 1 │ transforms │ ImageClassification │      0 │ train │     0 │
│ 2 │ train_acc  │ MulticlassAccuracy  │      0 │ train │     0 │
│ 3 │ val_acc    │ MulticlassAccuracy  │      0 │ train │     0 │
│ 4 │ test_acc   │ MulticlassAccuracy  │      0 │ train │     0 │
└───┴────────────┴─────────────────────┴────────┴───────┴───────┘

Trainable params: 5.1 K                                                                                            
Non-trainable params: 11.2 M                                                                                       
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44.727                                                                     
Modules in train mode: 72                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=5` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.8160000443458557     │
│         test_loss         │    0.5328184962272644     │
└───────────────────────────┴───────────────────────────┘

In [ ]:
csci4052u.report.summarize(all_versions=True)

                                        Training Summary                                        
┏━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┓
┃ Model  ┃ Version   ┃ Train Acc ┃ Val Acc ┃ Test Acc ┃ Trainable ┃ Frozen ┃ Epochs ┃ Duration ┃
┡━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━┩
│ ResNet │ version_0 │    0.8218 │  0.7980 │   0.8160 │      5.1K │  11.2M │      5 │   4m 10s │
└────────┴───────────┴───────────┴─────────┴──────────┴───────────┴────────┴────────┴──────────┘